# ER1: No-Comm Control Baseline

**Objective:** Establish the performance floor by training agents with **zero communication** on the Discovery K-N convergence task. Any communication method (ER2-E1) must beat this to justify its complexity.

**What this isolates:** Pure task difficulty under partial observability — agents can only sense via LiDAR, with no message passing between them.

**How to use this notebook:**
1. Set `PROFILE = "fast"` for a single-config run (100 training iterations, ~15 min) or `"complete"` for the full sweep (120 runs)
2. Run all cells top to bottom
3. Results are saved to `results/er1/YYYYMMDD_HHMM__<run_id>/` with structured subfolders

**How success (M1) is measured in the Discovery scenario:**
- Each target requires **K agents within `covering_range`** of it, all at the **same time** in a single step
- When a target gets covered, it is removed from the world (moved off-screen)
- The episode ends (`done=True`) when **every** target has been covered at least once
- **M1 = fraction of episodes that finished before `max_steps`**
- This is NOT "1 target covered" — it is "ALL targets covered", which requires sustained coordination
- With `targets_respawn=False`, the scenario tracks `all_time_covered_targets` and `done()` returns `all_time_covered_targets.all()`

## 1. Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
RENDEZVOUS_ROOT = REPO_ROOT / "rendezvous_comm"
sys.path.insert(0, str(RENDEZVOUS_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment
from src.display import display_config, display_metrics, display_sweep_summary, display_run_status
from src.storage import ExperimentStorage
from src.runner import run_sweep, evaluate_with_vmas, make_heuristic_policy_fn
from src.plotting import plot_sweep_heatmap, plot_seed_variance, save_figure

print(f"Torch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

Torch: 2.10.0  |  CUDA: False


## 2. Configuration

Set `PROFILE` below to control experiment scope:
- **`"fast"`**: 1 configuration (4 agents, MAPPO, seed 0), 6M training frames = **100 iterations**. Takes ~15 min on CPU.
- **`"complete"`**: Full sweep — N ∈ {2,3,4,6} × lidar ∈ {0.25, 0.35, 0.45} × {MAPPO, IPPO} × 5 seeds = **120 runs**.

All parameters are loaded from `configs/er1_no_comm.yaml` and the profile applies overrides on top.
The config uses `targets_respawn=False` so that episodes end when all targets are covered — this makes M1 (success rate) and M3 (steps to completion) meaningful.

In [2]:
PROFILE = "fast"  # "fast" or "complete"

spec = load_experiment(RENDEZVOUS_ROOT / "configs" / "er1_no_comm.yaml", profile=PROFILE)
spec.ensure_dirs()

display_config(spec, profile=PROFILE)

## 3. Sanity Check — Heuristic & Random Baselines

Before training, we verify the environment works and establish two reference points:
- **Heuristic**: The discovery scenario has a built-in `HeuristicPolicy` (circular patrol + greedy target-seeking). This is the non-learned upper reference.
- **Random**: Agents take random actions. This is the absolute floor.

### How covering works in Discovery
Each target requires **K agents within `covering_range`** simultaneously. When a target is covered:
- With `targets_respawn=False`: the target is removed, episode ends when ALL targets covered
- With `targets_respawn=True`: the target respawns at a new position (episode never ends)

We use `targets_respawn=False` so success means "agents coordinated to cover every target at least once".

### Metrics Reference
| Metric | ID | Meaning |
|--------|-----|---------|
| Success Rate | M1 | Fraction of episodes where **all** targets were covered before `max_steps` |
| Avg Targets/Step | M1b | Mean targets covered per step (for tracking partial progress) |
| Avg Return | M2 | Mean cumulative reward = covering reward + collision penalty + time penalty |
| Avg Steps | M3 | Mean steps until all targets covered (`max_steps` if episode didn't complete) |
| Collisions/Episode | M4 | Mean agent-agent collisions per episode |
| Tokens/Episode | M5 | Communication tokens used (always 0 for ER1 — no comm) |

In [3]:
SANITY_OVERRIDES = {
    "n_agents": 4, "n_targets": 7, "agents_per_target": 2,
    "targets_respawn": False,
}

heuristic_fn = make_heuristic_policy_fn()
heuristic_metrics = evaluate_with_vmas(
    spec.task, task_overrides=SANITY_OVERRIDES,
    policy_fn=heuristic_fn, n_eval_episodes=200, n_envs=200,
)
display_metrics(heuristic_metrics, title="Heuristic Baseline (4 agents, 7 targets, K=2)")

random_metrics = evaluate_with_vmas(
    spec.task, task_overrides=SANITY_OVERRIDES,
    policy_fn=None, n_eval_episodes=200, n_envs=200,
)
display_metrics(random_metrics, title="Random Baseline (4 agents, 7 targets, K=2)")

## 4. Training

Runs the parameter sweep defined by the config + profile. For each run the pipeline:
1. Saves a frozen config snapshot to `input/config.yaml`
2. Trains the policy using BenchMARL (MAPPO/IPPO with MLP network)
3. Logs all output to `logs/run.log` (timestamps, metrics, errors)
4. **Saves the trained policy** to `output/policy.pt` (can be reloaded later)
5. Evaluates the trained policy in deterministic mode
6. Saves metrics to `output/metrics.json`
7. Generates `report.txt` with full config, results, and artifact listing

BenchMARL's own training curves and checkpoints go to `output/benchmarl/`.

**How training works:**
- Each "iteration" collects `on_policy_collected_frames_per_batch` frames (60K) from `n_envs` parallel environments
- The policy is updated using PPO with `n_minibatch_iters` SGD epochs per batch
- The fast profile runs **100 iterations** (6M frames), complete runs **167 iterations** (10M frames)
- `skip_complete=True` means completed runs are detected and skipped on re-run

In [4]:
display_run_status(spec)

results = run_sweep(spec, skip_complete=True)

2026-03-09 09:30:48 INFO     START  er1_mappo_n4_t7_k2_l035_s0
2026-03-09 09:30:48 INFO       Algorithm:     mappo
2026-03-09 09:30:48 INFO       Seed:          0
2026-03-09 09:30:48 INFO       Frames:        6,000,000
2026-03-09 09:30:48 INFO       Device:        cpu
2026-03-09 09:30:48 INFO       Task overrides: {'n_agents': 4, 'n_targets': 7, 'agents_per_target': 2, 'lidar_range': 0.35}
2026-03-09 09:30:48 INFO       Output dir:    /Users/afin/Documents/Studio/PHD/Code/VectorizedMultiAgentSimulator/rendezvous_comm/results/er1/20260309_0930__er1_mappo_n4_t7_k2_l035_s0
/Users/afin/Documents/Studio/PHD/Code/VectorizedMultiAgentSimulator/.venv/lib/python3.11/site-packages/torchrl/collectors/_base.py:1045: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(
  0%|          | 0/100 [00:00<?, ?it/s]

2026-03-09 09:30:56,108 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([60000]) shape [END]


2026-03-09 09:31:25.019 Python[53215:5747867] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/qf/45hpwykd6s58js26xgtbw3tm0000gn/T/org.python.python.savedState
/Users/afin/Documents/Studio/PHD/Code/VectorizedMultiAgentSimulator/.venv/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
mean return = 0.289483904838562: 100%|██████████| 100/100 [1:00:12<00:00, 36.13s/it] 
2026-03-09 10:31:02 INFO     TRAINING COMPLETE  er1_mappo_n4_t7_k2_l035_s0  (1h 0m 12s)
2026-03-09 10:31:02 INFO     Saving trained policy...
2026-03-09 10:31:02 INFO       Policy saved to /Users/afin/Documents/Studio/PHD/Co

## 5. Results

Each completed run produces a timestamped folder with the following structure:
```
results/er1/YYYYMMDD_HHMM__<run_id>/       (e.g., 20260309_0030__er1_mappo_n4_t7_k2_l035_s0)
  input/
    config.yaml           # Frozen config: all task, training, and sweep params
  logs/
    run.log               # Full training log with timestamps
  output/
    benchmarl/            # Raw BenchMARL artifacts:
      <experiment_name>/
        csv/              #   Training curves (reward, loss, entropy, etc.)
        videos/           #   Evaluation videos at each eval interval
        checkpoints/      #   Policy checkpoints
    metrics.json          # Final evaluation metrics (M1-M5)
    policy.pt             # Trained policy weights (for export/reload)
  report.txt              # Human-readable summary: config, training, results, artifacts
```

**To reload a trained policy** for further evaluation or visualization:
```python
from src.runner import build_experiment
from src.storage import ExperimentStorage
storage = ExperimentStorage("er1")
run = storage.get_run("er1_mappo_n4_t7_k2_l035_s0")
state_dict = run.load_policy_state_dict()
# Rebuild experiment with same config, then: experiment.policy.load_state_dict(state_dict)
```

**Key metrics in `metrics.json`:**
- **M1 (Success Rate):** Fraction of eval episodes where ALL 7 targets were covered by K=2 agents each. This requires agents to coordinate spatially without communication.
- **M2 (Avg Return):** Cumulative reward. Positive = covering reward outweighs penalties. Negative = agents failing to cover targets or colliding too much.
- **M3 (Avg Steps):** How quickly agents complete. 200 = never completed (hit max_steps).
- **M4 (Collisions):** Agent-agent collisions. Shows whether agents learned spatial awareness.
- **M5 (Tokens):** Always 0 for ER1 (no communication channel exists).

In [5]:
storage = ExperimentStorage("er1")
all_metrics = storage.load_all_metrics()

if not all_metrics:
    print("No completed runs yet. Run section 4 first.")
else:
    print(f"Loaded {len(all_metrics)} completed runs\n")
    display_sweep_summary(all_metrics)

    # Show the report from the latest run
    latest_run_id = sorted(all_metrics.keys())[-1]
    run_storage = storage.get_run(latest_run_id)
    report_path = run_storage.run_dir / "report.txt"
    if report_path.exists():
        print(f"\n{'=' * 70}")
        print(f"  Report: {run_storage.run_dir.name}/report.txt")
        print(f"{'=' * 70}")
        print(report_path.read_text())

    # Check if policy is saved
    if run_storage.has_policy():
        policy_size = (run_storage.output_dir / "policy.pt").stat().st_size
        print(f"  Trained policy saved: {policy_size / 1024:.1f} KB")

Loaded 1 completed runs



,Success Rate,Avg Return,Avg Steps,Collisions/Episode
run_id,,,,
er1_mappo_n4_t7_k2_l035_s0,100.0%,67.17,100.0,0.00



  Report: 20260309_0930__er1_mappo_n4_t7_k2_l035_s0/report.txt
  EXPERIMENT REPORT
  Experiment:    ER1 - No-Comm Control
  Run ID:        er1_mappo_n4_t7_k2_l035_s0
  Folder:        20260309_0930__er1_mappo_n4_t7_k2_l035_s0
  Generated:     2026-03-09 10:33:25

  Description:
    Isolates baseline task difficulty by removing ALL communication. Establishes the floor that any comm method must beat. Train IPPO + MAPPO with zero comm channels.

----------------------------------------------------------------------------
  TASK CONFIGURATION (Discovery Scenario)
----------------------------------------------------------------------------

    n_agents (N)                           = 4 *      Number of agents
    n_targets (T)                           = 7 *      Number of targets to cover
    agents_per_target (K)                           = 2 *      Agents required per target simultaneously
    covering_range                               = 0.25        Distance threshold to 'cover' a tar

## 6. Analysis

Visualizations to understand the results:
- **6a. Heatmap:** Success rate across agent count and LiDAR range — shows which configurations are easiest/hardest.
- **6b. Algorithm comparison:** MAPPO vs IPPO with seed variance — checks if one algorithm consistently outperforms.
- **6c. Summary table:** All core metrics grouped by algorithm with mean and standard deviation.

These plots are saved to the experiment results directory for use in papers/reports.

In [6]:
df = storage.to_dataframe()

if df.empty:
    print("No data for plots. Complete section 4 first.")
else:
    # 6a. Success rate heatmap
    if len(df["n_agents"].unique()) > 1 if "n_agents" in df.columns else False:
        fig = plot_sweep_heatmap(
            df, metric="M1_success_rate",
            row_param="n_agents", col_param="lidar_range",
            title="ER1 No-Comm: Success Rate (N agents x LiDAR range)",
        )
        save_figure(fig, str(spec.results_dir / "er1_success_heatmap.png"))
        plt.show()
    else:
        print("Heatmap requires multiple agent counts (use 'complete' profile).")

Heatmap requires multiple agent counts (use 'complete' profile).


In [7]:
if not df.empty:
    # 6b. Algorithm comparison
    if "algorithm" in df.columns and len(df["algorithm"].unique()) > 1:
        fig = plot_seed_variance(
            df, metric="M1_success_rate", group_by="algorithm",
            title="ER1: MAPPO vs IPPO Success Rate (across seeds)",
        )
        save_figure(fig, str(spec.results_dir / "er1_algo_comparison.png"))
        plt.show()
    else:
        print("Algorithm comparison requires multiple algorithms (use 'complete' profile).")

    # 6c. Summary table
    print("\n" + "=" * 70)
    print("  SUMMARY TABLE: Core Metrics by Algorithm")
    print("=" * 70)
    group_cols = [c for c in ["algorithm"] if c in df.columns]
    if group_cols:
        summary = df.groupby(group_cols).agg({
            "M1_success_rate": ["mean", "std"],
            "M2_avg_return": ["mean", "std"],
            "M3_avg_steps": ["mean", "std"],
            "M4_avg_collisions": ["mean", "std"],
        }).round(4)
        from IPython.display import display
        display(summary)
    else:
        for key in ["M1_success_rate", "M2_avg_return", "M3_avg_steps", "M4_avg_collisions"]:
            vals = df[key].values
            print(f"  {key:<35} {vals.mean():.4f} +/- {vals.std():.4f}")

Algorithm comparison requires multiple algorithms (use 'complete' profile).

  SUMMARY TABLE: Core Metrics by Algorithm


M1_success_rate     M2_avg_return     M3_avg_steps      \
                     mean std          mean std         mean std   
algorithm                                                          
mappo                 1.0 NaN         67.17 NaN        100.0 NaN   

          M4_avg_collisions      
                       mean std  
algorithm                        
mappo                   0.0 NaN

## 7. Pass/Fail Assessment

**Expected outcome:** The no-comm floor should be in the **20-50% success range** for N=4, K=2. This means:
- If success is **below 20%**: The task may be too hard for any method — consider easier configs.
- If success is **20-50%**: Floor established. Communication methods (ER2-E1) must beat this.
- If success is **above 50%**: Task may be too easy without comm — consider harder configs (more targets, smaller LiDAR).

This assessment determines whether the experimental design is sound before investing compute in communication experiments.

In [8]:
if not df.empty:
    # Average success across all runs (or filter to N=4 if available)
    if "n_agents" in df.columns:
        n4_df = df[df["n_agents"] == 4]
    else:
        n4_df = df

    if not n4_df.empty:
        avg_success = n4_df["M1_success_rate"].mean()
        avg_return = n4_df["M2_avg_return"].mean()
        avg_steps = n4_df["M3_avg_steps"].mean()

        print("=" * 70)
        print(f"  N=4, K=2 Average Success Rate:  {avg_success:.1%}")
        print(f"  N=4, K=2 Average Return:        {avg_return:.2f}")
        print(f"  N=4, K=2 Average Steps:         {avg_steps:.0f}")
        print("=" * 70)

        if 0.20 <= avg_success <= 0.50:
            print("\n  PASS: Floor established in expected range (20-50%).")
            print("  Proceed to ER2+ experiments. Comm methods must beat this floor.")
        elif avg_success > 0.50:
            print("\n  NOTE: Floor higher than expected (>50%). Consider:")
            print("    - Increasing n_targets or agents_per_target")
            print("    - Reducing lidar_range")
        else:
            print(f"\n  NOTE: Floor lower than expected (<20%). Consider:")
            print("    - Training for more frames (current: {:,})".format(spec.train.max_n_frames))
            print("    - Checking if the task is too hard for this config")
            if PROFILE == "fast":
                print("    - Running with PROFILE='complete' for full training")
    else:
        print("No N=4 runs found.")
else:
    print("No data available. Run training first (section 4).")

  N=4, K=2 Average Success Rate:  100.0%
  N=4, K=2 Average Return:        67.17
  N=4, K=2 Average Steps:         100

  NOTE: Floor higher than expected (>50%). Consider:
    - Increasing n_targets or agents_per_target
    - Reducing lidar_range
